# Reconstructed trajectories and Lévy-like search statistics

This notebook reproduces the trajectory-scale results from publication-safe, local-coordinate reconstructions. Each compact archive contains only a floating-point `xyz_m` array. It does not contain raw sensor channels, absolute coordinates, source identifiers, or the private reconstruction objects.

The wording **Lévy-like** is intentional: heavy-tailed step lengths and superdiffusive displacement are descriptive signatures, not proof of a unique or optimal generative search rule.

In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd

ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").exists()
)
sys.path.insert(0, str(ROOT / "src"))

from trajectory.analysis import analyze_release
from trajectory.figures import plot_mu_distribution, plot_timescale_msd

INDEX = ROOT / "data/dswp/trajectory_index.csv"
index = pd.read_csv(INDEX)
index[["trajectory_id", "individual_id", "n_samples", "max_depth_m"]].head()

## Public reconstruction boundary

The index records anonymous trajectory and individual tokens, the relative archive path, sampling rate, reviewed foraging bounds, and non-identifying quality summaries. The 228 distributed paths are the numeric release form of pitch-adaptive reconstructions built upstream by the internal `notebooks/00_save_3d_reconstructions.ipynb` workflow. The private shared folder and its identifiers are provenance inputs, not public dependencies, and are therefore not embedded here.

Seventeen recorded dives did not have an available pitch-adaptive reconstruction and are absent from the path bundle. The manuscript analysis further applies its maximum-depth and minimum-sample/step criteria, yielding 209 analysed foraging dives.

In [ ]:
release_overview = {
    "released_trajectories": int(len(index)),
    "released_individual_tokens": int(index["individual_id"].nunique()),
    "total_samples": int(index["n_samples"].sum()),
}
release_overview

## Production analysis choices

- keep the released trajectories at full `float64` precision, because reducing precision can move nearly collinear samples across the turning threshold;
- identify turns above 30 degrees and compute distances only between successive detected turning points (endpoints are not injected);
- fit a continuous power-law tail by scanning 30 candidate lower cutoffs from the 5th to 80th percentiles, with at least 30 tail steps;
- calculate three-dimensional time-averaged MSD over 50 logarithmically spaced lags; and
- analyse the longest non-surface segment for the reported short-, intermediate-, and long-lag crossover.

These choices match the result-producing Lévy code rather than a generic replacement workflow.

In [ ]:
per_dive, population, timescale = analyze_release(INDEX)
pd.Series(population, name="value").to_frame()

Expected headline values are 209 dives from 20 individual tokens; median dive-level $\mu=2.3543$; median individual-level $\mu=2.3637$; pooled $\mu=2.1546$; and median whole-foraging MSD exponent $\alpha=1.9090$. Every included dive is superdiffusive ($\alpha>1$).

## Scale-dependent directional persistence

The crossover analysis asks how the MSD exponent changes with lag. Near-ballistic short-lag motion gives way to weaker—but still superdiffusive—persistence at longer lags.

In [ ]:
timescale_columns = [column for column in timescale if column.startswith("alpha_")]
timescale[timescale_columns].median().rename("median_alpha").to_frame()

In [ ]:
ax_mu = plot_mu_distribution(per_dive)
ax_mu.set_title("Dive-level step-length exponents")
ax_msd = plot_timescale_msd(timescale)
ax_msd.set_title("Scale-dependent MSD exponents")

The production figures and CSV/JSON summaries are written by `python scripts/run_pipeline.py`. Numerical summaries—not pixel identity—are the authoritative reproducibility target across plotting backends.